In [1]:
import pandas as pd
import re

# ── 파일 로드 ──────────────────────────────────────────
FILE_PATH = r"D:\Project\★아파트지도\data\공동주택현황.csv"

for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
    try:
        df = pd.read_csv(FILE_PATH, encoding=enc)
        print(f"✅ 인코딩: {enc} | 총 {len(df):,}행")
        break
    except:
        continue

# ── 도로명주소 추출: '(' 앞 텍스트 ────────────────────────
df['도로명주소'] = df['주소'].str.split('(').str[0].str.strip()

# ── 품질 체크 항목 ─────────────────────────────────────
checks = {}

# 1. 주소 자체가 비어있는 행
checks['① 주소 결측'] = df['주소'].isna() | (df['주소'] == '')

# 2. '(' 없어서 도로명 추출 못한 경우 (원본 = 추출값이면 괄호 없음)
checks['② 괄호 없음'] = ~df['주소'].str.contains(r'\(', na=False)

# 3. 도로명주소가 너무 짧음 (10자 미만)
checks['③ 주소 너무 짧음'] = df['도로명주소'].str.len() < 10

# 4. 도로명 키워드 없음 (로·길·대로 모두 없음)
road_kw = r'로\b|길\b|대로\b'
checks['④ 도로명 키워드 없음'] = ~df['도로명주소'].str.contains(road_kw, na=False)

# 5. 시도 키워드 없음 (광역시·도·시 등)
sido_kw = r'광역시|특별시|특별자치시|특별자치도|\b[가-힣]{2,3}도\b|\b[가-힣]{2,3}시\b'
checks['⑤ 시/도 키워드 없음'] = ~df['도로명주소'].str.contains(sido_kw, na=False)

# 6. 숫자(건물번호) 없음
checks['⑥ 건물번호 없음'] = ~df['도로명주소'].str.contains(r'\d', na=False)

# ── 결과 요약 출력 ─────────────────────────────────────
print("\n📋 품질 체크 결과")
print("="*50)
problem_masks = {}
for name, mask in checks.items():
    cnt = mask.sum()
    pct = cnt / len(df) * 100
    problem_masks[name] = mask
    status = "🔴" if cnt > 0 else "✅"
    print(f"{status} {name}: {cnt:,}건 ({pct:.1f}%)")

# ── 문제 행 샘플 출력 ──────────────────────────────────
print("\n🔍 문제 행 샘플 (각 항목 최대 5행)")
print("="*50)
for name, mask in problem_masks.items():
    sub = df[mask][['아파트명', '주소', '도로명주소']].head(5)
    if len(sub) > 0:
        print(f"\n▶ {name}")
        print(sub.to_string(index=False))

# ── 전체 문제 행 별도 저장 ─────────────────────────────
any_problem = pd.concat(problem_masks.values(), axis=1).any(axis=1)
df_problems = df[any_problem].copy()
df_problems['문제항목'] = ', '.join([])  # 아래서 채움
df_problems['문제항목'] = df[any_problem].apply(
    lambda row: ', '.join(
        [name for name, mask in problem_masks.items() if mask[row.name]]
    ), axis=1
)
OUT = r"D:\Project\★아파트지도\data\주소_문제행.csv"
df_problems.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f"\n💾 문제 행 저장 완료: {OUT} ({len(df_problems):,}행)")

# ── 정상 주소 미리보기 ─────────────────────────────────
print("\n✅ 정상 도로명주소 샘플 10건")
print(df[~any_problem]['도로명주소'].head(10).to_string())

✅ 인코딩: cp949 | 총 3,263행

📋 품질 체크 결과
🔴 ① 주소 결측: 1건 (0.0%)
🔴 ② 괄호 없음: 5건 (0.2%)
🔴 ③ 주소 너무 짧음: 224건 (6.9%)
🔴 ④ 도로명 키워드 없음: 5건 (0.2%)
🔴 ⑤ 시/도 키워드 없음: 3,009건 (92.2%)
🔴 ⑥ 건물번호 없음: 1건 (0.0%)

🔍 문제 행 샘플 (각 항목 최대 5행)

▶ ① 주소 결측
   아파트명  주소 도로명주소
합     계 NaN   NaN

▶ ② 괄호 없음
      아파트명                 주소              도로명주소
캠프워커패밀리하우징         남구 봉덕동 350         남구 봉덕동 350
      무열마을      수성구 만촌동 산 184      수성구 만촌동 산 184
캠프워커패밀리하우징         남구 봉덕동 350         남구 봉덕동 350
대가야스카이뷰203 고령군 대가야읍 장기리 258-2 고령군 대가야읍 장기리 258-2
   합     계                NaN                NaN

▶ ③ 주소 너무 짧음
         아파트명                    주소     도로명주소
       청라힐스자이  중구 남산로 73 (남산동 3049) 중구 남산로 73
        U-스퀘어  북구 영송로 36 (태전동 1215) 북구 영송로 36
힐스테이트데시앙도남2단지 북구 도남길 16 (국우동 592-1) 북구 도남길 16
      앞산태왕아너스  남구 효서길 53 (봉덕동 1642) 남구 효서길 53
      앞산서한이다음  남구 신촌길 94 (봉덕동 1637) 남구 신촌길 94

▶ ④ 도로명 키워드 없음
      아파트명                 주소              도로명주소
캠프워커패밀리하우징         남구 봉덕동 350         남구 봉덕동 350
      무열마을      수성구 만촌동 산 184      수

In [3]:
"""
SGIS OpenAPI 지오코딩 파이프라인
- Access Token 자동 발급
- 도로명주소 추출 (괄호 앞 텍스트)
- 지오코딩 실행 (rate limit 대응, 실패 재시도)
- EPSG:5179 → WGS84 좌표 변환 (pyproj)
- 중간 저장 (SAVE_EVERY 건마다)
- 결과 저장 (성공 / 실패 분리)
"""

import pandas as pd
import requests
import time
from pyproj import Transformer

# ── 설정 ───────────────────────────────────────────────────────
FILE_PATH    = r"D:\Project\★아파트지도\data\공동주택현황.csv"
OUT_SUCCESS  = r"D:\Project\★아파트지도\data\공동주택현황_지오코딩완료.csv"
OUT_FAIL     = r"D:\Project\★아파트지도\data\공동주택현황_지오코딩실패.csv"
OUT_PROGRESS = r"D:\Project\★아파트지도\data\공동주택현황_지오코딩진행중.csv"

SERVICE_ID  = "02db299fb553470b97aa"
SECRET_KEY  = "381beff8ef174fee8691"

DELAY       = 0.15   # 요청 간격 (초)
MAX_RETRY   = 3      # 실패 시 최대 재시도 횟수
RETRY_DELAY = 2.0    # 재시도 대기 (초)
SAVE_EVERY  = 50     # N건마다 중간 저장

# ── EPSG:5179 → WGS84 변환기 ────────────────────────────────────
transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)

def tm_to_wgs84(x: float, y: float):
    lon, lat = transformer.transform(x, y)
    return round(lat, 7), round(lon, 7)

# ── 1. 파일 로드 ────────────────────────────────────────────────
print("=" * 60)
print("📂 파일 로드 중...")
df = None
for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
    try:
        df = pd.read_csv(FILE_PATH, encoding=enc)
        print(f"   ✅ 인코딩: {enc} | 총 {len(df):,}행 x {len(df.columns)}열")
        print(f"   컬럼: {df.columns.tolist()}")
        break
    except:
        continue

# ── 2. 도로명주소 추출 ──────────────────────────────────────────
df['도로명주소'] = df['주소'].str.split('(').str[0].str.strip()
print(f"\n📌 도로명주소 추출 샘플 (상위 5건):")
for i, row in df[['아파트명', '주소', '도로명주소']].head(5).iterrows():
    print(f"   [{i+1}] {row['아파트명']}")
    print(f"        원본  : {row['주소']}")
    print(f"        추출  : {row['도로명주소']}")

# ── 3. Access Token 발급 ────────────────────────────────────────
def get_access_token(service_id, secret_key):
    url = "https://sgisapi.mods.go.kr/OpenAPI3/auth/authentication.json"
    params = {"consumer_key": service_id, "consumer_secret": secret_key}
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if data.get("errCd") != 0:
        raise RuntimeError(f"Token 발급 실패: {data}")
    token = data["result"]["accessToken"]
    print(f"\n🔑 Access Token 발급 완료: {token[:16]}...")
    return token

token = get_access_token(SERVICE_ID, SECRET_KEY)

# ── 4. 단건 지오코딩 함수 ───────────────────────────────────────
def geocode(address, access_token):
    url = "https://sgisapi.mods.go.kr/OpenAPI3/addr/geocode.json"
    params = {
        "accessToken": access_token,
        "address": address,
        "pagenum": 0,
        "resultcount": 1,
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    if data.get("errCd") != 0:
        return None

    resultdata = data.get("result", {}).get("resultdata", [])
    if not resultdata:
        return None

    r = resultdata[0]
    x_tm = float(r.get("x", 0))
    y_tm = float(r.get("y", 0))
    lat, lon = tm_to_wgs84(x_tm, y_tm)

    return {
        "lat":       lat,
        "lon":       lon,
        "sido_nm":   r.get("sido_nm", ""),
        "sgg_nm":    r.get("sgg_nm", ""),
        "adm_nm":    r.get("adm_nm", ""),
        "addr_type": r.get("addr_type", ""),
    }

# ── 5. 전체 지오코딩 실행 ───────────────────────────────────────
print(f"\n{'='*60}")
print(f"🌐 지오코딩 시작 | 총 {len(df):,}건 | {SAVE_EVERY}건마다 중간저장")
print(f"{'='*60}")

# 테스트: 첫 3건 미리 찍어보기
print("\n🧪 첫 3건 테스트 실행:")
for i, row in df.head(3).iterrows():
    try:
        r = geocode(row['도로명주소'], token)
        if r:
            print(f"   [{i+1}] ✅ {row['도로명주소']}")
            print(f"        → 위도: {r['lat']}, 경도: {r['lon']} | {r['sido_nm']} {r['sgg_nm']} {r['adm_nm']} | 유형: {r['addr_type']}")
        else:
            print(f"   [{i+1}] ❌ {row['도로명주소']} → 결과 없음")
        time.sleep(DELAY)
    except Exception as e:
        print(f"   [{i+1}] ⚠️  오류: {e}")

print(f"\n{'─'*60}")
ans = input("▶ 테스트 결과 확인됐으면 Enter로 전체 실행, q 입력 시 중단: ").strip()
if ans.lower() == 'q':
    print("중단합니다.")
    exit()

lats, lons, sido_nms, sgg_nms, adm_nms, addr_types, statuses = [], [], [], [], [], [], []
fail_count  = 0
start_time  = time.time()

for idx, row in df.iterrows():
    addr = row['도로명주소']
    seq  = idx + 1

    result = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            result = geocode(addr, token)
            break
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 401:
                print(f"\n   ⚠️  [{seq}] Token 만료 → 재발급 중...")
                token = get_access_token(SERVICE_ID, SECRET_KEY)
            else:
                print(f"\n   ⚠️  [{seq}] HTTP 오류 ({e.response.status_code}) — {attempt}/{MAX_RETRY} 재시도")
                time.sleep(RETRY_DELAY)
        except Exception as e:
            print(f"\n   ⚠️  [{seq}] 오류 ({e}) — {attempt}/{MAX_RETRY} 재시도")
            time.sleep(RETRY_DELAY)

    if result:
        lats.append(result["lat"])
        lons.append(result["lon"])
        sido_nms.append(result["sido_nm"])
        sgg_nms.append(result["sgg_nm"])
        adm_nms.append(result["adm_nm"])
        addr_types.append(result["addr_type"])
        statuses.append("성공")
    else:
        lats.append(None)
        lons.append(None)
        sido_nms.append("")
        sgg_nms.append("")
        adm_nms.append("")
        addr_types.append("")
        statuses.append("실패")
        fail_count += 1
        print(f"   ❌ [{seq:>5}] 실패: {addr}")

    # ── 진행률 + 중간저장 (SAVE_EVERY건마다) ──────────────────
    if seq % SAVE_EVERY == 0 or seq == len(df):
        elapsed   = time.time() - start_time
        remaining = (elapsed / seq) * (len(df) - seq)
        success   = seq - fail_count
        rate      = success / seq * 100

        print(f"\n   📊 [{seq:>5}/{len(df):,}] "
              f"성공: {success:,} ({rate:.1f}%) | 실패: {fail_count:,} | "
              f"경과: {elapsed/60:.1f}분 | 잔여: {remaining/60:.1f}분")

        # 현재까지 결과를 df에 붙여서 중간 저장
        df_partial = df.iloc[:seq].copy()
        df_partial['위도']         = lats
        df_partial['경도']         = lons
        df_partial['시도명']       = sido_nms
        df_partial['시군구명']     = sgg_nms
        df_partial['읍면동명']     = adm_nms
        df_partial['주소유형']     = addr_types
        df_partial['지오코딩상태'] = statuses
        df_partial.to_csv(OUT_PROGRESS, index=False, encoding='utf-8-sig')
        print(f"   💾 중간저장 완료 → {OUT_PROGRESS}")

    time.sleep(DELAY)

# ── 6. 최종 결과 컬럼 추가 ─────────────────────────────────────
df['위도']         = lats
df['경도']         = lons
df['시도명']       = sido_nms
df['시군구명']     = sgg_nms
df['읍면동명']     = adm_nms
df['주소유형']     = addr_types
df['지오코딩상태'] = statuses

# ── 7. 저장 ────────────────────────────────────────────────────
df_success = df[df['지오코딩상태'] == '성공']
df_fail    = df[df['지오코딩상태'] == '실패']

df_success.to_csv(OUT_SUCCESS, index=False, encoding='utf-8-sig')
df_fail.to_csv(OUT_FAIL,    index=False, encoding='utf-8-sig')

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ 완료! 총 소요시간: {total_time/60:.1f}분")
print(f"   성공: {len(df_success):,}건 → {OUT_SUCCESS}")
print(f"   실패: {len(df_fail):,}건  → {OUT_FAIL}")

# 주소유형별 통계
print(f"\n📈 주소유형별 통계 (addr_type):")
print(f"   1=도로명완전매칭, 2=지번, 6=행정구역 수준")
print(df_success['주소유형'].value_counts().to_string())

📂 파일 로드 중...
   ✅ 인코딩: cp949 | 총 3,263행 x 11열
   컬럼: ['아파트명', '아파트코드', '세대수', '우편번호', '주소', '난방구분', '용도', '공동주택계약', '설치위치', '서비스센터', 'Unnamed: 10']

📌 도로명주소 추출 샘플 (상위 5건):
   [1] 전기재료관
        원본  : 북구 유통단지로 38 (산격동 1665)
        추출  : 북구 유통단지로 38
   [2] 범어아이파크2차 1BL
        원본  : 수성구 동대구로46길 33 (범어동 650)
        추출  : 수성구 동대구로46길 33
   [3] 힐스테이트칠성더오페라
        원본  : 북구 중앙대로 506-3 (칠성동2가 321)
        추출  : 북구 중앙대로 506-3
   [4] 이안엑소디움에이펙스
        원본  : 달서구 달구벌대로 1514 (감삼동 599-51)
        추출  : 달서구 달구벌대로 1514
   [5] 대구죽전행복주택LH아파트
        원본  : 달서구 와룡로53길 47 (죽전동 74-23)
        추출  : 달서구 와룡로53길 47

🔑 Access Token 발급 완료: f06c0ee1-b785-44...

🌐 지오코딩 시작 | 총 3,263건 | 50건마다 중간저장

🧪 첫 3건 테스트 실행:
   [1] ✅ 북구 유통단지로 38
        → 위도: 35.9056908, 경도: 128.6072819 | 대구광역시 북구 산격2동 | 유형: 6
   [2] ✅ 수성구 동대구로46길 33
        → 위도: 35.8506946, 경도: 128.6274125 | 대구광역시 수성구 범어1동 | 유형: 6
   [3] ✅ 북구 중앙대로 506-3
        → 위도: 35.8782136, 경도: 128.5956255 | 대구광역시 북구 칠성동 | 유형: 6

──────────────────────────────────────

In [4]:
import pandas as pd
import json
from shapely.geometry import shape, Point

FILE_CSV     = r"D:\Project\★아파트지도\data\공동주택현황_지오코딩완료.csv"
FILE_GEOJSON = r"D:\Project\★아파트지도\data\고시지역.geojson"
FILE_OUT     = r"D:\Project\★아파트지도\data\공동주택현황_최종.csv"

for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
    try:
        df = pd.read_csv(FILE_CSV, encoding=enc)
        break
    except:
        continue

with open(FILE_GEOJSON, encoding='utf-8') as f:
    gj = json.load(f)

polygons = [(feat['properties']['Name'], shape(feat['geometry'])) for feat in gj['features']]

def find_gosi(lon, lat):
    pt = Point(lon, lat)
    for name, geom in polygons:
        if geom.contains(pt):
            return name
    return None

df['고시지역명'] = df.apply(lambda r: find_gosi(r['경도'], r['위도']), axis=1)
df['고시여부']   = df['고시지역명'].apply(lambda x: '예' if x else '아니오')

df.to_csv(FILE_OUT, index=False, encoding='utf-8-sig')

print(f"완료 → {FILE_OUT}")
print(df['고시지역명'].value_counts(dropna=False))

완료 → D:\Project\★아파트지도\data\공동주택현황_최종.csv
고시지역명
None    3173
성서지구      45
대곡지구      16
용산지구      13
죽곡지구       8
장기지구       4
Name: count, dtype: int64
